# Preparación de datos

Responsable: Harry Vargas

Usuario: hvarga4

Repositorio: https://github.com/hvarga4/be_SIRC_2022

## Paquetes y funciones

In [1]:
import os
import re
import yaml
import time
from typing import List
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import warnings
from pandas_profiling import ProfileReport
from tqdm import tqdm

warnings.filterwarnings("ignore")

# Conexion con fuentes de Datos de Vertex en CDS
from datetime import datetime as dt, date, timedelta
from google.cloud import bigquery
from google.cloud import aiplatform, storage
import google.cloud.bigquery.magics


# Configuracion pandas
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 1000)
sns.set_theme(
    palette="bright",
    rc={"figure.dpi": 100, "savefig.dpi": 300, "figure.figsize": (5, 5)},
)

In [2]:
def read_gcp_bigquery_table(
    proyecto: str,
    dataset: str,
    table: str,
    columnas: List[str],
    columna_fecha: str,
    fecha_inicio: str,
    fecha_fin: str,
) -> pd.DataFrame:

    """
    Funcion de lectura de tablas alojadas en GCP BigQuery

    Parameters
    ----------
    proyecto: str
        Nombre del proyecto de GCP donde se aloja BigQuery
    dataset: str
        Nombre del dataset o del esquema de almacenamiento en GCP BigQuery
    table:str
        Nombre de la tabla alojada en GCP BigQuery
    columnas: List[str]
        Lista de columnas a extraer en la consulta
    columna_fecha: str
        Nombre de la columna que contiene la informacion de la fecha de captura de los datos
    fecha_inicio: str
        Fecha de inicio en formato '%Y-%m-%d' o 'YYYY-MM-dd' de captura de datos
    fecha_fin: srt
        Fecha de fin en formato '%Y-%m-%d' o 'YYYY-MM-dd' de captura de datos

    Returns
    -------
    pd.DataFrame
        Conjunto de datos con la informacion de la tabla extraida

    """
    # Definicion del query de extraccion
    if columna_fecha == None:
        query = f"""SELECT {', '.join(columnas)}
                FROM {proyecto}.{dataset}.{table};
                """
    else:
        query = f"""SELECT {', '.join(columnas)}
                FROM {proyecto}.{dataset}.{table}
                WHERE {columna_fecha} BETWEEN '{fecha_inicio}' AND '{fecha_fin}';
                """

    # Apertura del cliente de GCP BigQuery
    with bigquery.Client(project=proyecto) as client:
        df = client.query(query).result().to_dataframe()

    return df

## Carga de bases

In [3]:
# Definición de rutas

os.environ["PROYECT_GCP"] = "bdb-gcp-cds-pr-ac-ba"
os.environ["BUCKET_PROYECT"] = "bdb-gcp-cds-pr-ac-ba-analitica-avanzada/banca-empresas"
os.environ["STORAGE_PATH"] = "gs://{}".format(os.environ["BUCKET_PROYECT"])

os.environ["PROYECT_PATH"] = os.path.join(os.environ["STORAGE_PATH"], "639_Retencion")
os.environ["REPORTS_PATH"] = os.path.join(os.environ["PROYECT_PATH"], "results")
os.environ["DATA_PATH"] = os.path.join(os.environ["PROYECT_PATH"], "data")
os.environ["DATA_RAW"] = os.path.join(os.environ["DATA_PATH"], "raw")
os.environ["DATA_INTER"] = os.path.join(os.environ["DATA_PATH"], "interim")

In [4]:
# %%time
# #Parámetros de lectura

# read_params = yaml.safe_load(
#     Path("/home/jupyter/be_SIRC_2022/conf/base/parameters/read.yml").read_text()
# )

# # Lectura de datos desde BigQuery
# datos = {}
# for dataframe, params in tqdm(read_params.items()):
#     datos[dataframe] = read_gcp_bigquery_table(**params)

# fecha_corte = "agosto_2023"
# # Escritura de datos en Cloud Storage
# for  data_name, data in tqdm(datos.items()):
#     datos[data_name].to_parquet(f'{os.environ["DATA_RAW"]}/{data_name}_{fecha_corte}.parquet')
#     print(f'{data_name}_{fecha_corte} guardado')

In [6]:
# Lectura de datos
fecha_corte = "agosto_2023"

# No varían en el tiempo
sectores = pd.read_excel(os.environ["DATA_RAW"] + "/be-sectores-CIIU-julio2021.xlsx")
# semaforo = pd.read_parquet(
#     os.environ["DATA_RAW"] + "/be-semaforo-20220826.parquet"
# )

# Se actualiza manual - archivo en ruta analytics
color = pd.read_excel(
    os.environ["DATA_RAW"] + "/historico_color_sector_corte_agosto_2023.xlsx"
)
# Datos actualizados corte diciembre 2022
asignacion = pd.read_parquet(
    os.environ["DATA_RAW"] + f"/asignacion_{fecha_corte}.parquet"
)
emis = pd.read_parquet(os.environ["DATA_RAW"] + f"/emis_diciembre_2022.parquet")
cash = pd.read_parquet(os.environ["DATA_RAW"] + f"/cash_{fecha_corte}.parquet")
pasivo_promedio = pd.read_parquet(
    os.environ["DATA_RAW"] + f"/deposito_promedio_{fecha_corte}.parquet"
)
pasivo_saldo = pd.read_parquet(
    os.environ["DATA_RAW"]
    + f"/deposito_saldo_diciembre_2022.parquet"  # no se tiene que actualizar
)
apercan = pd.read_parquet(os.environ["DATA_RAW"] + f"/apercan_{fecha_corte}.parquet")
# cartera_promedio = pd.read_parquet(
#     os.environ["DATA_RAW"] + f"/cartera_promedio_{fecha_corte}.parquet"
# )
cartera = pd.read_parquet(os.environ["DATA_RAW"] + f"/cartera_{fecha_corte}.parquet")

## Preparación de datos

In [8]:
# Asignacion

col_types = {"Cliente_Id": "object"}
asignacion = asignacion.astype(col_types)

asignacion.dropna(subset=["Segmento"], inplace=True)

asignacion.sort_values(by=["Cliente_Id", "identificador"], inplace=True)
asignacion = asignacion.drop_duplicates(subset=["Cliente_Id"], keep="last")
dic_zona = {
    "Oficial Oriente": "ORIENTE",
    "Oficial Occidente": "OCCIDENTE",
    "Antioquia Corporativo": "ANTIOQUIA",
    "Nororiente Corporativo": "NORORIENTE",
    "Oficial Antioquia": "ANTIOQUIA",
    "Oficial Central": "CENTRAL",
    "Bogotá Empresarial 2": "BOGOTA",
    "Occidente Corporativo": "OCCIDENTE",
    "Occidente Empresarial": "OCCIDENTE",
    "Oficial Costa 1": "COSTA",
    "Oficial Costa 2": "COSTA",
    "Bogotá Corporativo": "BOGOTA",
    "Antioquia Empresarial": "ANTIOQUIA",
    "Nororiente Empresarial": "NORORIENTE",
    "Bogotá Norte Mediana": "BOGOTA",
    "Bogotá Sur Mediana": "BOGOTA",
    "Occidente Mediana": "OCCIDENTE",
    "Bogotá Empresarial 1": "BOGOTA",
    "Costa Mediana": "COSTA",
    "Bogotá Empresarial 3": "BOGOTA",
    "Bogotá Centro Mediana": "BOGOTA",
    "Oriente Mediana": "ORIENTE",
    "Centroriente Mediana": "CENTROORIENTE",
    "Antioquia Mediana": "ANTIOQUIA",
}

asignacion["Zona"] = asignacion["Zona"].replace(dic_zona)

In [9]:
color.columns = [str(i) for i in color.columns]

# Hacer un melt para cambiar estructura de datos
color_sector = pd.melt(
    color,
    id_vars=["MACRO SECTOR", "SECTOR"],
    value_vars=color.columns.drop(["MACRO SECTOR", "SECTOR"]).to_list(),
    var_name="Fecha",
    value_name="Color_Sector",
)

# Recodificar valores para el color
campos_color_sector = {
    "NA": np.nan,
    "Sector no existente en el pasado": np.nan,
    0: np.nan,
    "Naranja1": "Naranja",
    "Naranja2": "Naranja",
    "VERDE": "Verde",
    "AMARILLO": "Amarillo",
    "NARANJA": "Naranja",
    "ROJO": "Rojo",
}

color_sector["Color_Sector"] = color_sector["Color_Sector"].replace(campos_color_sector)

# Castear fecha
color_sector["Fecha_Corte_Anio_Mes"] = (
    pd.to_datetime(color_sector["Fecha"]).dt.to_period("M").astype("str")
)

# Quitar variable vieja de fecha
color_sector.drop(columns=["Fecha"], inplace=True)

color_sector.rename(
    columns={"MACRO SECTOR": "Macrosector", "SECTOR": "Sector"}, inplace=True
)

In [10]:
# # Semaforo

# semaforo["Fecha_Corte_Anio_Mes"] = (
#     pd.to_datetime(semaforo["Fecha_Corte"]).dt.to_period("M").astype("str")
# )
# sector_cliente = semaforo.drop_duplicates(subset=["Sector", "Cliente_Id"])[
#     ["Sector", "Cliente_Id"]
# ]
# color_cliente = (
#     semaforo.drop_duplicates(subset=["Sector", "Fecha_Corte", "Color_Sector"])[
#         ["Sector", "Fecha_Corte", "Color_Sector"]
#     ]
#     .sort_values(by=["Sector", "Fecha_Corte"])
#     .dropna(subset=["Color_Sector"])
# )

In [11]:
# Cash
# Casteo de la fecha
cash["Fecha_Corte_Anio_Mes"] = (
    pd.to_datetime(cash["Fecha_Corte"]).dt.to_period("M").astype("str")
)

# Agregación mensual todos los productos
cash = (
    cash.groupby(by=["Cliente_Id", "Fecha_Corte_Anio_Mes", "Producto"])
    .agg(
        TRX=("TRX", "sum"),
        VAL=("VAL", "sum"),
        Familia=("Familia", "first"),
    )
    .reset_index()
)


# Canales
canales = (
    cash[cash.Familia.isin(["Canales"])]
    .pivot(
        index=["Cliente_Id", "Fecha_Corte_Anio_Mes"],
        columns=["Producto"],
        values=["TRX"],
    )
    .reset_index()
)

canales.columns = ["Cliente_Id", "Fecha_Corte_Anio_Mes", 
                   # "TRX_Afinidad", 
                   "TRX_Portal"]

# Pagos y Recaudos, quitar Depósitos
cash = cash.loc[
    (cash.Familia.isin(["Pagos", "Recaudos"]))
    & ~(cash.Producto.isin(["Corriente", "Ahorros"]))
]

# VAL, sum o promedio
cash = (
    cash.groupby(by=["Cliente_Id", "Fecha_Corte_Anio_Mes"])
    .agg(
        cash_sum_trx=("TRX", "sum"),
        cash_sum_val=("VAL", "sum"),
        cash_cant_prod=("Producto", "nunique"),
    )
    .reset_index()
)
# Se crea campo para identificación del producto
cash["cash"] = 1

In [12]:
# Saldos Pasivo
# Filtrar moneda extranjera
# Filtrar total pasivos y Total Pasivos

pasivo_saldo = pasivo_saldo.loc[
    (pasivo_saldo.Moneda == "COP Millones")
    & (pasivo_saldo.Producto_Detalle != "Total Pasivos")
    & (pasivo_saldo.Fecha_Corte.astype("str").str.contains("2019"))
]

# Castear fecha y Cliente_Id
pasivo_saldo["Fecha_Corte_Anio_Mes"] = (
    pd.to_datetime(pasivo_saldo["Fecha_Corte"]).dt.to_period("M").astype("str")
)

pasivo_saldo.Cliente_Id = pasivo_saldo.Cliente_Id.astype("object")

# Cantidad de productos con saldo mayor a cero
pasivo_activo_2019 = (
    pasivo_saldo[pasivo_saldo.Valor > 0]
    .groupby(by=["Cliente_Id", "Fecha_Corte_Anio_Mes"])
    .agg(dep_cant_prod=("Producto", "nunique"))
)

# Agregación mensual
# Suma saldos promedio al mes
pasivo_agg_2019 = (
    pasivo_saldo.groupby(by=["Cliente_Id", "Fecha_Corte_Anio_Mes"])
    .agg(dep_sum_val=("Valor", "sum"))
    .reset_index()
)

# Unir campos
depositos_2019 = pasivo_agg_2019.merge(
    pasivo_activo_2019, on=["Cliente_Id", "Fecha_Corte_Anio_Mes"], how="left"
)

In [13]:
# Saldos Pasivo Promedio
# Filtrar moneda extranjera
# Filtrar total pasivos y Total Pasivos

pasivo_promedio = pasivo_promedio.loc[
    (pasivo_promedio.Producto != "ME")
    & (pasivo_promedio.Producto_Detalle != "Total Pasivos")
]

# Castear fecha y Cliente_Id
pasivo_promedio["Fecha_Corte_Anio_Mes"] = (
    pd.to_datetime(pasivo_promedio["Fecha_Corte"]).dt.to_period("M").astype("str")
)

pasivo_promedio.Cliente_Id = pasivo_promedio.Cliente_Id.astype("object")

# Filtrar registros con saldo mayor a cero
pasivo_activo = (
    pasivo_promedio[pasivo_promedio.Valor > 0]
    .groupby(by=["Cliente_Id", "Fecha_Corte_Anio_Mes"])
    .agg(dep_cant_prod=("Producto", "nunique"))
)

# Agregación mensual
# Suma saldos promedio al mes
pasivo_agg = (
    pasivo_promedio.groupby(by=["Cliente_Id", "Fecha_Corte_Anio_Mes"])
    .agg(dep_sum_val=("Valor", "sum"))
    .reset_index()
)

# Unir campos
depositos = pasivo_agg.merge(
    pasivo_activo, on=["Cliente_Id", "Fecha_Corte_Anio_Mes"], how="left"
)


# depositos_2019 y depositos contienen toda la información desde 2019 a 2022
pasivo = pd.concat([depositos_2019, depositos])

# se crea campo para identificación del producto
pasivo["deposito"] = 1

In [14]:
# Saldos promedios cartera

# Quitar nulos
cartera.dropna(subset=["Cliente_Id"], inplace=True)

# Casteo de columnas
cartera.Cliente_Id = cartera.Cliente_Id.astype("object")
cartera["Fecha_Corte_Anio_Mes"] = (
    pd.to_datetime(cartera["Fecha_Corte"]).dt.to_period("M").astype("str")
)

# Quitar Producto agregado y quitar Moneda Extranjera
cartera = cartera.loc[
    (cartera.Producto != "Total Cartera De Creditos Moneda Legal")
    & (cartera.Familia != "Cartera ME")
]

# Agregación deuda mensual en todos los productos
# suma o promedio??

cartera = (
    cartera.groupby(by=["Cliente_Id", "Fecha_Corte_Anio_Mes"])
    .agg(cart_sum_val=("Saldo", "sum"), cart_cant_prod=("Producto", "nunique"))
    .reset_index()
)

cartera["cartera_saldo"] = 1

In [15]:
## APERCAN
# algo raro pasa, nuevo archivo de apercan es demasiado pesado y su procesamiento está tomando demasiado tiempo

# Quitar nulos
apercan.dropna(subset=["Cliente_Id"], inplace=True)

# Creación Fecha anio-mes
apercan["Fecha_Corte_Anio_Mes"] = (
    pd.to_datetime(apercan["Fecha_Corte"]).dt.to_period("M").astype("str")
)

# Excluir moneda extranjera
apercan = apercan.query("Familia != 'Cartera ME'")

# Agrupar cliente mes por pagos y desembolsos
apercan_agg = pd.pivot_table(
    apercan,
    index=["Cliente_Id", "Fecha_Corte_Anio_Mes"],
    columns=["TRN"],
    values=["Valor"],
    aggfunc="sum",
    fill_value=0,
).reset_index()

apercan_agg.columns = [
    "Cliente_Id",
    "Fecha_Corte_Anio_Mes",
    "cart_sum_desembolso",
    "cart_sum_pago",
]
# desembolsos - pagos
apercan_agg["cart_neto"] = (
    apercan_agg["cart_sum_desembolso"] - apercan_agg["cart_sum_pago"]
)
apercan_agg["apercan"] = 1

In [16]:
# EMIS Datos cierre de 31 diciembre 2021
# Emis cierre de 2020 hay que cambiar nombres

emis.dropna(subset=["Cliente_Id_Sin_DV"], inplace=True)
emis["Cliente_Id_Sin_DV"] = emis["Cliente_Id_Sin_DV"].astype("int").astype("object")

# Se parte en dos el campo, se guarda el importante en una nueva variable y se elimina el campo original
emis["Empleados"] = (
    emis["Numero_de_empleados"]
    .astype("str")
    .str.split(n=2, expand=True)[0]
    .str.replace(",", "")
)
emis.drop(columns=["Numero_de_empleados"], inplace=True)
emis.Empleados = emis.Empleados.replace("None", np.nan).astype("float")

emis_vars = [
    "Cliente_Id_Sin_DV",
    "Activos_Totales",
    "Total_Ingreso_Operativo",
    "Ciudad",
    "Total_de_patrimonio",
    "Pasivos_Totales",
    "ROA_Operativo__",
    "Margen_Operacional__",
    "Rendimiento_Sobre_Los_Activos_ROA__",
    "Rendimiento_Sobre_El_Patrimonio_ROE__",
    "Ganancia_Perdida_Neta",
    "Razon_De_Liquidez_x",
    "Rotacion_Del_Capital_De_Trabajo_x",
    "Margen_Neto__",
    "Ganancia_operativa_EBIT",
    "Empleados",
]

emis = emis[emis_vars]

## Exploración preliminar

## Joins para consolidar tabla master

In [17]:
# Fechas límite de la master
fechas = pd.DataFrame(
    {
        "Fecha_Corte_Anio_Mes": pd.period_range(
            "2019-01", end="2023-08", freq="M"
        ).strftime("%Y-%m")
    }
)

# Se crea un esqueleto con la estructura
var_asignacion = [
    "Cliente_Id",
    "Cliente_Nombre",
    "Tipo_Cliente",
    "Clasificacion_Epc",
    "CIIU",
    "Relacion",
    "Numero_Empresas",
    "Segmento",
    "Subsegmento",
    "Zona",
    "Domicilio",
]

df = asignacion[var_asignacion].merge(fechas, how="cross")

In [18]:
# Join para obtener Sector por cliente
df = df.merge(
    sectores[["BCIIU_CODIGO", "SECTOR_CIIU"]],
    left_on="CIIU",
    right_on="BCIIU_CODIGO",
    how="left",
)

df.rename(columns={"SECTOR_CIIU": "Sector"}, inplace=True)

In [19]:
# Join con sectores y colores
df = df.merge(
    color_sector,
    how="left",
    left_on=["Fecha_Corte_Anio_Mes", "Sector"],
    right_on=["Fecha_Corte_Anio_Mes", "Sector"],
)

df = df.drop(columns=["BCIIU_CODIGO"])

# Checkpoint 1
df.to_parquet(os.environ["DATA_INTER"] + "/master_raw_1.parquet")

In [20]:
# Join con matriz cash
df = df.merge(cash, on=["Cliente_Id", "Fecha_Corte_Anio_Mes"], how="left")

# Join con información de canales

df = df.merge(canales, on=["Cliente_Id", "Fecha_Corte_Anio_Mes"], how="left")

# Checkpoint 2
df.to_parquet(os.environ["DATA_INTER"] + "/master_raw_2.parquet")

In [21]:
# Join con depósitos
df = df.merge(pasivo, on=["Cliente_Id", "Fecha_Corte_Anio_Mes"], how="left")

# Checkpoint 3
df.to_parquet(os.environ["DATA_INTER"] + "/master_raw_3.parquet")

In [22]:
# Join con información de cartera
df = df.merge(cartera, on=["Cliente_Id", "Fecha_Corte_Anio_Mes"], how="left")

df = df.merge(apercan_agg, on=["Cliente_Id", "Fecha_Corte_Anio_Mes"], how="left")

df["cartera"] = df[["apercan", "cartera_saldo"]].apply(
    lambda x: 1 if (x[0] == 1) | (x[1] == 1) else 0, axis=1
)

In [23]:
# Join con Emis, se modifica para cruzar con CLiente ID sin Digito de verificación

df["Cliente_Id_Sin_DV"] = (
    df["Cliente_Id"].astype("str").str[0:9].astype("int").astype("object")
)

df = df.merge(emis, how="left", on="Cliente_Id_Sin_DV")

# Checkpoint 4
df.to_parquet(os.environ["DATA_INTER"] + "/master_raw_4.parquet")

In [24]:
df.info()

# integrar Margen de Contribución y CIFIN
# Supersociedades

<class 'pandas.core.frame.DataFrame'>
Int64Index: 5772088 entries, 0 to 5772087
Data columns (total 47 columns):
 #   Column                                 Dtype  
---  ------                                 -----  
 0   Cliente_Id                             object 
 1   Cliente_Nombre                         object 
 2   Tipo_Cliente                           object 
 3   Clasificacion_Epc                      object 
 4   CIIU                                   float64
 5   Relacion                               int64  
 6   Numero_Empresas                        float64
 7   Segmento                               object 
 8   Subsegmento                            object 
 9   Zona                                   object 
 10  Domicilio                              int64  
 11  Fecha_Corte_Anio_Mes                   object 
 12  Sector                                 object 
 13  Macrosector                            object 
 14  Color_Sector                           object 
 15

## SANBOX

In [25]:
# df = pd.read_parquet(os.environ["DATA_INTER"] + "/master_raw_4.parquet")

In [29]:
# df[df.Fecha_Corte_Anio_Mes == '2023-08']

,Cliente_Id,Cliente_Nombre,Tipo_Cliente,Clasificacion_Epc,CIIU,Relacion,Numero_Empresas,Segmento,Subsegmento,Zona,Domicilio,Fecha_Corte_Anio_Mes,Sector,Macrosector,Color_Sector,cash_sum_trx,cash_sum_val,cash_cant_prod,cash,TRX_Portal,dep_sum_val,dep_cant_prod,deposito,cart_sum_val,cart_cant_prod,cartera_saldo,cart_sum_desembolso,cart_sum_pago,cart_neto,apercan,cartera,Cliente_Id_Sin_DV,Activos_Totales,Total_Ingreso_Operativo,Ciudad,Total_de_patrimonio,Pasivos_Totales,ROA_Operativo__,Margen_Operacional__,Rendimiento_Sobre_Los_Activos_ROA__,Rendimiento_Sobre_El_Patrimonio_ROE__,Ganancia_Perdida_Neta,Razon_De_Liquidez_x,Rotacion_Del_Capital_De_Trabajo_x,Margen_Neto__,Ganancia_operativa_EBIT,Empleados
55,3383,SANABRIA HECTOR JOSE,Actual,Clave,129.0,1,NaN,PYME,Py1,Bogotá 1,1294,2023-08,EXPLOTACIONES AGROPECUARIAS,AGROPECUARIO Y PESCA,Verde,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,3383,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
111,4831,BULLA ORJUELA HERNANDO,Actual,Preferente,4669.0,0,16.0,Mediana,Mediana,BOGOTA,1176,2023-08,COMERCIO EN GENERAL AL POR MENOR,CADENAS MAYORISTAS Y MINORISTAS,Verde,12.0,188690484.0,3.0,1.0,NaN,2879.053703,2.0,1.0,0.023,13.0,1.0,NaN,NaN,NaN,NaN,1,4831,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
167,9599,RUEDA FULA PABLO ELIAS,Inactivo,Inactivo,90.0,1,NaN,PYME,Py2,Bogotá Sur,1661,2023-08,INVERSIONISTA,INVERSIONISTA,Naranja,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,9599,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
223,49347,GUARIN MARTINEZ DARIO,Actual,Preferente,10.0,0,5.0,Corporativo,Corporativo,NORORIENTE,1747,2023-08,ASALARIADOS,ACTIVIDADES VARIAS,Naranja,NaN,NaN,NaN,NaN,NaN,12827.746150,2.0,1.0,0.000,13.0,1.0,NaN,NaN,NaN,NaN,1,49347,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
279,68638,CARREAO TORRES LUIS FRANCISCO,Inactivo,Inactivo,10.0,1,NaN,OIS,Institucional,OIS,1783,2023-08,ASALARIADOS,ACTIVIDADES VARIAS,Naranja,NaN,NaN,NaN,NaN,NaN,0.008450,1.0,1.0,0.000,13.0,1.0,NaN,NaN,NaN,NaN,1,68638,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5771863,97061223952,GUALDRON SOGAMOSO LAURA NATALY,Potencial,Potencial,10.0,1,NaN,Mediana,Mediana,COSTA,1191,2023-08,ASALARIADOS,ACTIVIDADES VARIAS,Naranja,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,970612239,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5771919,98031454379,AGUILAR BERMUDEZ NATHALIA ANDREA,Potencial,Potencial,10.0,1,NaN,Mediana,Mediana,ORIENTE,1180,2023-08,ASALARIADOS,ACTIVIDADES VARIAS,Naranja,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,980314543,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5771975,98052671000,DORIA IZASA JOSE ANGEL,Potencial,Potencial,10.0,1,NaN,Mediana,Mediana,COSTA,1192,2023-08,ASALARIADOS,ACTIVIDADES VARIAS,Naranja,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.092,13.0,1.0,NaN,NaN,NaN,NaN,1,980526710,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5772031,99061701814,LOPEZ RIVERA LAURA VALENTINA,Potencial,Potencial,10.0,1,NaN,Mediana,Mediana,BOGOTA,1176,2023-08,ASALARIADOS,ACTIVIDADES VARIAS,Naranja,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,990617018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
# id_col = "Cliente_Id"
# time_series_col = "cart_sum_pago"
# period_length = 3

# diff_periods = "diff_periods"
# mean_first_period_col = "mean_first_period"
# mean_sec_period_col = "mean_sec_period"


# def compute_diff_periods(
#     # params: Dict[str, Any],
#     df: pd.DataFrame,
# ):
#     # Definir parámetros
#     #     id_col = 'Cliente_Id'
#     #     time_series_col = "cart_sum_pago"
#     #     period_length = 3

#     #     diff_periods = "diff_periods"
#     #     mean_first_period_col = "mean_first_period"
#     #     mean_sec_period_col = "mean_sec_period"

#     df["val_cum_sum_0"] = np.nan  # primer mes de mi periodo anterior
#     df["val_cum_sum_1"] = np.nan  # último mes de mi periodo anterior

#     df["val_cum_sum_2"] = np.nan  # primer mes de mi periodo actual
#     df["val_cum_sum_3"] = df.groupby(by=id_col)[
#         time_series_col
#     ].cumsum()  # último mes del periodo actual

#     # 2 meses de periodicidad y comparación periodos consecutivos:
#     # febrero mes actual
#     # cumsum(feb) - cumsum(ene) -> promedio de val trx para el periodo actual
#     # cumsum(dic) - cumsum(nov) -> promedio de val trx para el periodo anterior
#     # promedio(periodo actual) - promedio(periodo anterior) -> diff periodos

#     df["val_cum_sum_2"] = df.groupby(by=id_col)["val_cum_sum_3"].shift(
#         period_length - 1
#     )

#     df["val_cum_sum_1"] = df.groupby(by=id_col)["val_cum_sum_3"].shift(period_length)

#     df["val_cum_sum_0"] = df.groupby(by=id_col)["val_cum_sum_3"].shift(
#         2 * period_length - 1
#     )

#     df[mean_first_period_col] = (
#         df["val_cum_sum_1"] - df["val_cum_sum_0"]
#     ) / period_length
#     df[mean_sec_period_col] = (
#         df["val_cum_sum_3"] - df["val_cum_sum_2"]
#     ) / period_length
#     df[diff_periods] = (df[mean_sec_period_col] / df[mean_first_period_col]) - 1
#     #   df.drop(
#     #         columns=[
#     #             "val_cum_sum_0",
#     #             "val_cum_sum_1",
#     #             "val_cum_sum_2",
#     #             "val_cum_sum_3",

#     #         ],
#     #         inplace=True,
#     #     )

#     return df